In [ ]:
    !pip install -q transformers==4.41.2 tokenizers==0.19.1 bitsandbytes accelerate==0.26.0 einops timm torchvision pillow matplotlib

In [ ]:
import torch
import requests
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from transformers import AutoProcessor, AutoModelForCausalLM
import transformers.dynamic_module_utils as dyn_utils
from unittest.mock import patch

# 🚨 [두더지 1번 완벽 제압] flash_attn 요구사항 몰래 삭제하기
orig_get_imports = dyn_utils.get_imports

def custom_get_imports(filename):
    imports = orig_get_imports(filename)
    if "flash_attn" in imports:
        imports.remove("flash_attn")
    return imports
# ==========================================================

model_id = "microsoft/Florence-2-base"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"⏳ {device} 환경에서 Florence-2 모델을 불러오는 중...")

# 🚨 [두더지 2번 완벽 제압] 안정적인 구버전 환경 + 수문장 패치 동시 적용!
with patch("transformers.dynamic_module_utils.get_imports", custom_get_imports):
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True).to(device)

print("✅ 모델 로딩 완벽 성공!\n")

# 2. 테스트용 거실 사진 불러오기
image_path = "/content/image(7).png"
image = Image.open(image_path).convert("RGB")

# 3. 사용자 입력받기 (예: chair, lamp, table)
user_input = input("🤖 찾고 싶은 물건을 영어로 입력하세요 (예: chair): ")

# 4. 프롬프트 및 추론
task_prompt = "<CAPTION_TO_PHRASE_GROUNDING>"
text_input = task_prompt + user_input

inputs = processor(text=text_input, images=image, return_tensors="pt").to(device)

print(f"\n🔍 '{user_input}'(을)를 찾는 중...")
with torch.no_grad():
    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        num_beams=3
    )

# 5. 결과 파싱 및 시각화
generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
parsed_answer = processor.post_process_generation(generated_text, task=task_prompt, image_size=image.size)

fig, ax = plt.subplots(1, figsize=(10, 8))
ax.imshow(image)

results = parsed_answer.get(task_prompt, {})
bboxes = results.get('bboxes', [])
labels = results.get('labels', [])

if not bboxes:
    print(f"❌ 화면에서 '{user_input}'(을)를 찾을 수 없습니다.")
else:
    print(f"🎯 총 {len(bboxes)}개의 '{user_input}'(을)를 찾았습니다!\n")

    for box, label in zip(bboxes, labels):
        x1, y1, x2, y2 = box
        center_x = (x1 + x2) / 2
        center_y = (y1 + y2) / 2

        print(f"[{label}] 바운딩 박스: ({x1:.1f}, {y1:.1f}) ~ ({x2:.1f}, {y2:.1f})")
        print(f" └─> 🤖 로봇 목표 좌표(Pick): X={center_x:.1f}, Y={center_y:.1f}\n")

        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        ax.plot(center_x, center_y, 'ro')
        ax.text(x1, y1 - 10, label, color='red', fontsize=12, weight='bold', backgroundcolor='white')

plt.axis('off')
plt.show()

In [ ]:
import torch
from PIL import Image
import matplotlib.pyplot as plt
from transformers import AutoProcessor, AutoModelForCausalLM
import transformers.dynamic_module_utils as dyn_utils
from unittest.mock import patch

# 🚨 [수문장 무력화 패치]
orig_get_imports = dyn_utils.get_imports
def custom_get_imports(filename):
    imports = orig_get_imports(filename)
    if "flash_attn" in imports:
        imports.remove("flash_attn")
    return imports
# ==========================================================

model_id = "microsoft/Florence-2-base"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"⏳ {device} 환경에서 Florence-2 모델을 불러오는 중...")

with patch("transformers.dynamic_module_utils.get_imports", custom_get_imports):
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True).to(device)

print("✅ 모델 로딩 완벽 성공!\n")

# 2. 테스트용 사진 불러오기 (경로를 알맞게 수정하세요!)
image_path = "/content/image(7).png"
image = Image.open(image_path).convert("RGB")

# 3. [핵심 변경] 프롬프트를 '상세 묘사 모드'로 변경
task_prompt = "<MORE_DETAILED_CAPTION>"

# 묘사 모드에서는 뒤에 내가 찾는 단어를 붙이지 않고, 프롬프트 자체만 텍스트로 넘깁니다.
inputs = processor(text=task_prompt, images=image, return_tensors="pt").to(device)

print("\n🔍 사진을 뚫어져라 분석하며 상황을 추론하는 중...")
with torch.no_grad():
    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        num_beams=3
    )

# 4. 결과 파싱
generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
parsed_answer = processor.post_process_generation(generated_text, task=task_prompt, image_size=image.size)

# 5. [핵심 변경] 바운딩 박스를 그리지 않고, 추론된 문장만 깔끔하게 출력합니다.
print("\n========================================")
print("🧠 Florence-2의 상세 묘사(추론) 결과:")
print("========================================")
print(parsed_answer[task_prompt])
print("========================================\n")

# 사진은 참고용으로 화면에 띄워만 줍니다.
fig, ax = plt.subplots(1, figsize=(8, 6))
ax.imshow(image)
plt.axis('off')
plt.show()

In [ ]:
##라마

import torch
import time  # 시간 측정을 위해 추가
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import transformers.dynamic_module_utils as dyn_utils
from unittest.mock import patch
from google.colab import userdata

# 1. 신분증 확인
clean_token = userdata.get('HF_TOKEN').strip()

# 🚨 Florence-2 패치
orig_get_imports = dyn_utils.get_imports
def custom_get_imports(filename):
    imports = orig_get_imports(filename)
    if "flash_attn" in imports:
        imports.remove("flash_attn")
    return imports

device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. 모델 로딩
print("⏳ [시스템 가동] 로봇의 눈과 뇌를 연결 중입니다...")

# 눈 (Florence-2)
model_id = "microsoft/Florence-2-base"
with patch("transformers.dynamic_module_utils.get_imports", custom_get_imports):
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    vision_model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True).to(device)

# 뇌 (Llama 3 8B)
llm_id = "meta-llama/Meta-Llama-3-8B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("🧠 Llama 3 뇌 이식 중...")
llm_tokenizer = AutoTokenizer.from_pretrained(llm_id, token=clean_token)
llm_model = AutoModelForCausalLM.from_pretrained(
    llm_id,
    quantization_config=bnb_config,
    device_map={"": 0},  # 버그 방지를 위한 강제 GPU 할당
    token=clean_token
)
print("✅ 가동 준비 완료!\n")

# 3. 분석 시작 (시간 측정 포함)
image_path = "/content/image(7).png" 
image = Image.open(image_path).convert("RGB")

print("🔍 [현장 분석 시작]")
start_total = time.time() # 전체 시간 측정 시작

# --- Florence-2 추론 시작 ---
start_vision = time.time()
inputs = processor(text="<MORE_DETAILED_CAPTION>", images=image, return_tensors="pt").to(device)
with torch.no_grad():
    generated_ids = vision_model.generate(input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"], max_new_tokens=1024, num_beams=3)
florence_description = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
florence_description = processor.post_process_generation(florence_description, task="<MORE_DETAILED_CAPTION>", image_size=image.size)["<MORE_DETAILED_CAPTION>"]
end_vision = time.time()
# --- Florence-2 추론 종료 ---

print(f"👀 센서 데이터 수신 완료: {florence_description}")
print(f"⏱️ 시각 분석 소요 시간: {end_vision - start_vision:.2f}초\n")

# 4. Llama 3 판단 (시간 측정 및 한국어 강제)
print("🤔 [중앙 통제 AI 판단 중...]\n")

# 1. 상세 분석 지시 (User Question)
user_question = """Analyze the provided sensor description with professional EOD logic. Evaluate the physical components to determine the authenticity of the threat and issue a final resolution command.

Provide your report in the following format:
1. **Object Classification & Reasoning:** Categorize the object as [Benign / Hoax / Potential IED] and explain the reasoning based on visual evidence (e.g., presence of a power source, wiring complexity, or mundane materials).
2. **Threat Assessment:** Evaluate the risk level (Low to Critical). Identify if the device appears functional or if it is a non-working imitation designed to cause disruption.
3. **Tactical Directives:** Command a specific resolution action.
   - If Benign/Hoax: Command 'Manual Clearance' or 'Close-range Inspection'.
   - If Potential IED: Command 'Initiate Disruption Sequence' (e.g., deploying a disrupter or robotic manipulation) to neutralize the threat immediately."""

# 2. 역할 및 페르소나 설정 (Messages)
messages = [
    {
        "role": "system",
        "content": "You are the Tactical Intelligence Command (TIC) AI for an Explosive Ordnance Disposal (EOD) robot. Your mission is to analyze high-stakes sensor data to differentiate between harmless everyday objects, sophisticated hoaxes (fake devices), and actual Improvised Explosive Devices (IEDs). Your primary goal is to provide a decisive operational path for the robot to resolve the threat on-site through logical evidence and technical assessment."
    },
    {"role": "user", "content": f"Field Sensor Data: {florence_description}\n\nTask: {user_question}"}
]

# --- Llama 3 추론 시작 ---
start_llm = time.time()
prompt = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
llm_inputs = llm_tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = llm_model.generate(
        **llm_inputs,
        max_new_tokens=1024,
        temperature=0.1,  # 0.3보다 낮춰서 지시 이행률(한국어 유지)을 높임
        do_sample=True,
        eos_token_id=[llm_tokenizer.eos_token_id, llm_tokenizer.convert_tokens_to_ids("<|eot_id|>")]
    )

response = llm_tokenizer.decode(outputs[0][llm_inputs['input_ids'].shape[1]:], skip_special_tokens=True)
end_llm = time.time()
# --- Llama 3 추론 종료 ---

end_total = time.time() # 전체 시간 측정 종료

print("========================================")
print("🚨 AMR 최종 명령 (판단 결과) 🚨")
print("========================================")
print(response)
print("========================================")

# 📊 성능 리포트 출력 (포트폴리오용)
print(f"\n📈 [성능 분석 리포트]")
print(f"1. 시각 분석(Florence-2) 시간: {end_vision - start_vision:.2f}s")
print(f"2. 지능 판단(Llama 3 8B) 시간: {end_llm - start_llm:.2f}s")
print(f"3. 총 파이프라인 처리 시간: {end_total - start_total:.2f}s")
print(f"4. 사용 모델: Llama 3 8B Instruct (4-bit Quantized)")
print(f"📊 실시간 VRAM 점유량: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")